PROCESS

In [1]:
import sys, subprocess
print("Kernel:", sys.executable)

# install ke kernel aktif
subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "nltk"])

Kernel: d:\ANACONDA\python.exe


0

In [2]:
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "Sastrawi"])

0

In [3]:
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "xgboost"])

0

In [4]:
import pandas as pd
import re

1. CLEANING DATA

In [5]:
df = pd.read_csv('data_label.csv', sep=';')

df.head()

,text,label
0,Kunjungan Prabowo ini untuk meresmikan dan men...,Sumber Daya Alam
1,RT Anies dapat tepuk tangan meriah saat jadi R...,Politik
2,@CIqXqwGAT04tMtx4OCATxjoVq7vv/Y8HeYaIOgMFg8Y= ...,Demografi
3,RT @L3R8XFBw3WGbxRPSj0/0hHZTbqVGX7qtfwRg9zmhK7...,Politik
4,Anies Baswedan Harap ASN termasuk TNI dan Polr...,Politik


In [6]:
# cek duplikasi
df.shape

(5000, 2)

In [7]:
# drop semua yg duplikat
df = df.drop_duplicates(subset=['text'])
df.duplicated().sum()

np.int64(0)

In [8]:
df = df.dropna()

In [9]:
df.isnull().sum()

text     0
label    0
dtype: int64

In [10]:
df.shape

(4583, 2)

In [11]:
import pandas as pd
import re

def clean_twitter_text(text):
    text = str(text)
    
    # 1. Hapus RT dan [RE ...] pattern
    text = re.sub(r'\bRT\b', '', text)
    text = re.sub(r'\[RE[^\]]*\]', '', text)
    
    # 2. Hapus mention (termasuk mention encrypted panjang)
    text = re.sub(r'@\S+', '', text)
    
    # 3. Hapus URL
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    
    # 4. Hapus # tapi SIMPAN teksnya
    text = re.sub(r'#(\w+)', r'\1', text)
    
    # 5. Hapus semua karakter non-ASCII (encoding rusak, emoji broken)
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)
    
    # 6. Lowercase
    text = text.lower()
    
    # 7. Hapus karakter khusus, simpan huruf, angka, spasi
    text = re.sub(r'[^a-z0-9\s]', '', text)
    
    # 8. Normalisasi karakter berulang "bagussss" → "bagus"
    text = re.sub(r'(.)\1{2,}', r'\1', text)
    
    # 9. Rapikan spasi
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Apply ke dataframe
df['text'] = df['text'].apply(clean_twitter_text)

# Cek hasilnya
print("=== SEBELUM ===")
print(df['text'].head(5).values)
print("\n=== SESUDAH ===")
print(df['text'].head(5).values)

=== SEBELUM ===
['kunjungan prabowo ini untuk meresmikan dan menyerahkan proyek bantuan air bersih di lima titik indonesiasentris indonesiahijau 02melanjutkan anakmudaindonesiaemas prabowo subianto'
 'anies dapat tepuk tangan meriah saat jadi rektor mewajibkan mata kuliah antikorupsi untuk memutus mata rantai korupsi aminmiskinkankoruptor'
 'emng bener sih pendukung 01 ada yg goblok begitu jg dg pendukung 02 hnya sj menurut pak ridwan kamil skemanya terbalik klo 01 mayoritas pendidikan menengah atas artinya ada jg pendidikan rendah yg milih'
 'sewaktu anies bersikap kritis ke kinerja pak prabowo dianggap engga sopan karena dianggap kurang menghormati orang tua giliran skrg gibran yg tengil dan sok kritis malah dianggap kritis dan keras apakah ini tidak standar ganda'
 'anies baswedan harap asn termasuk tni dan polri pegang sumpahnya dalam pemilu']

=== SESUDAH ===
['kunjungan prabowo ini untuk meresmikan dan menyerahkan proyek bantuan air bersih di lima titik indonesiasentris indonesia

In [12]:
# 1. Ada gak hasil cleaning yang jadi kosong?
df[df['text'].str.strip() == '']



,text,label
74,,Sosial Budaya
943,,Sosial Budaya
1157,,Politik
1952,,Geografi
1970,,Sumber Daya Alam
2648,,Politik
3307,,Ekonomi
3891,,Politik
4783,,Sosial Budaya


In [13]:
df = df[df['text'].str.strip() != '']


In [14]:
df = df.reset_index(drop=True)


In [15]:
print(f"Data tersisa: {df.shape[0]} baris")
print(f"Cek kosong lagi: {(df['text'].str.strip() == '').sum()} baris")

Data tersisa: 4574 baris
Cek kosong lagi: 0 baris


In [16]:
# Distribusi panjang teks - penting untuk deteksi anomali
df['text'].str.len().describe()

count    4574.000000
mean      206.147355
std       256.997177
min         7.000000
25%       123.000000
50%       170.000000
75%       219.000000
max      4643.000000
Name: text, dtype: float64

In [17]:
# Lihat teks yang sangat pendek
df[df['text'].str.len() < 15][['text', 'label']].head(10)

,text,label
531,prabowo,Politik


In [18]:
# Lihat teks yang sangat panjang
df[df['text'].str.len() > 1000][['text', 'label']].head(100)

,text,label
64,konsisten selama ini bersuara melawan radikali...,Demografi
232,juru bicara tpn ganjarmahfud aryo seno bagasko...,Sosial Budaya
274,mantan ketua komnas ham apresiasi polri buka r...,Sosial Budaya
275,menjelang tengah malam ini aku mau ucapkan ter...,Politik
348,ganjarmahfud md program kuliah gratis bagi ana...,Demografi
...,...,...
3993,menjegal videotron anies baswedan hanya bagian...,Politik
4006,sampai kapan kubu prabowo bodohi rakyat oleh t...,Politik
4013,pemerintah harus menggandeng masyarakat dari a...,Sumber Daya Alam
4268,koalisi anies ganjar pada awalnya sih malu mal...,Politik


In [19]:
# Lihat distribusi label dari teks panjang ini
df[df['text'].str.len() > 1000]['label'].value_counts()

label
Politik                    28
Sosial Budaya              11
Ideologi                    9
Ekonomi                     7
Pertahanan dan Keamanan     3
Demografi                   2
Sumber Daya Alam            2
Name: count, dtype: int64

In [20]:
# Truncate teks yang lebih dari 500 karakter
df['text'] = df['text'].apply(lambda x: x[:500] if len(x) > 500 else x)

# Verifikasi
print(f"Max panjang sekarang: {df['text'].str.len().max()}")
print(f"Total data: {df.shape[0]}")

Max panjang sekarang: 500
Total data: 4574


In [21]:
# Rekap akhir sebelum lanjut preprocessing
print("=== REKAP CLEANING ===")
print(f"Total data bersih : {df.shape[0]} baris")
print(f"Distribusi label  :\n{df['label'].value_counts()}")
print(f"\nSample hasil cleaning:")
df[['text', 'label']].sample(5)

=== REKAP CLEANING ===
Total data bersih : 4574 baris
Distribusi label  :
label
Politik                    2949
Sosial Budaya               416
Ideologi                    339
Pertahanan dan Keamanan     330
Ekonomi                     308
Sumber Daya Alam            153
Demografi                    60
Geografi                     19
Name: count, dtype: int64

Sample hasil cleaning:


,text,label
3817,sederhana jelas dan efektif paslon no 03 capre...,Politik
245,rangkuman komentar terbaik netizen selengkapny...,Politik
3190,pengancam anies baswedan tertangkap polri pela...,Pertahanan dan Keamanan
1447,politisasi bansos menjelang pemilu bansos dite...,Politik
4189,gua nggak ingin 02 menang tapi kalau menang gu...,Politik


In [22]:
# Lihat semua data Geografi yang ada
df[df['label'] == 'Geografi']['text'].values

array(['malam tahun baru ganjar salawatan amp istighosah bareng puluhan ribu warga semarang senin 112024 dipimpin gus ali gondrong berlokasi di lapangan sapta rengga bandungan kabupaten semarang suasana penuh suka cita ganjar dan warga dalam menyambut tahun 2024 bersamasama',
       'jangan golput pilih capres no 3 ganjar pranowo mahfud md untuk masa depan yang lebih baik ganjar mahfud hebat l3bihbaik mahfudlebihbaik3 ganjarmahfud2024 guru ngaji satu juta',
       'di tiktok ada yg bilang anies kepengen bangun 40 kota seperti jakarta tapi provinsi cuma 34 ngakak',
       'kuliah gratis dampaknya bisa utk semua anak baik yg tinggal di kota maupun pelosok kalau makan siang gratis akan sulit diterapkan di pedalaman internet gratis akan membuat anakanak malah suka main game mager kurang aktivitas fisik',
       'owi dan orgnya ngerti pasti bkl panen cuan banyak soalnya yg di bawah ikn itu tanahnya prabowo sm adek prabowo bilangnya mah tanah negara tp wkwk pantes milih bangun 1 wilayah buka

In [23]:
# Lihat semua data Demografi (60 baris)
print(df['label'].value_counts())
print(f"\nRasio Politik:Geografi = {2949//19}:1")
print(f"Rasio Politik:Demografi = {2949//60}:1")

label
Politik                    2949
Sosial Budaya               416
Ideologi                    339
Pertahanan dan Keamanan     330
Ekonomi                     308
Sumber Daya Alam            153
Demografi                    60
Geografi                     19
Name: count, dtype: int64

Rasio Politik:Geografi = 155:1
Rasio Politik:Demografi = 49:1


AUGMENTASI

In [24]:
# Cek dulu kolom yang ada
print(df.columns.tolist())
print(df.head(2))

['text', 'label']
                                                text             label
0  kunjungan prabowo ini untuk meresmikan dan men...  Sumber Daya Alam
1  anies dapat tepuk tangan meriah saat jadi rekt...           Politik


In [ ]:
import random
import nltk
from nltk.corpus import wordnet
nltk.download('wordnet')
nltk.download('omw-1.4')

# FUNGSI EDA

def get_synonyms(word):
    synonyms = []
    for syn in wordnet.synsets(word, lang='ind'):
        for lemma in syn.lemmas(lang='ind'):
            if lemma.name() != word:
                synonyms.append(lemma.name().replace('_', ' '))
    return list(set(synonyms))

def random_swap(words, n=1):
    words = words.copy()
    for _ in range(n):
        if len(words) < 2:
            break
        i, j = random.sample(range(len(words)), 2)
        words[i], words[j] = words[j], words[i]
    return words

def random_delete(words, p=0.1):
    if len(words) == 1:
        return words
    return [w for w in words if random.random() > p]

def random_insert(words, n=1):
    words = words.copy()
    for _ in range(n):
        synonyms = []
        random.shuffle(words)
        for word in words:
            synonyms = get_synonyms(word)
            if synonyms:
                break
        if synonyms:
            insert_pos = random.randint(0, len(words))
            words.insert(insert_pos, random.choice(synonyms))
    return words

def augment_text(text, n_augment=5):
    words = text.split()
    augmented = []
    
    for _ in range(n_augment):
        technique = random.choice(['swap', 'delete', 'insert'])
        
        if technique == 'swap':
            new_words = random_swap(words)
        elif technique == 'delete':
            new_words = random_delete(words)
        else:
            new_words = random_insert(words)
            
        augmented.append(' '.join(new_words))
    
    return augmented

# AUGMENTASI GEOGRAFI & DEMOGRAFI

random.seed(42)

target = 200  # target jumlah per kelas

augmented_rows = []

for label_name in ['Geografi', 'Demografi']:
    class_df = df[df['label'] == label_name]
    current_count = len(class_df)
    needed = target - current_count
    
    print(f"\n{label_name}: {current_count} → target {target} (perlu tambah {needed})")
    
    texts = class_df['text'].tolist()
    
    while needed > 0:
        # Ambil teks random dari kelas ini
        base_text = random.choice(texts)
        
        # Generate augmentasi
        n_aug = min(5, needed)
        aug_texts = augment_text(base_text, n_augment=n_aug)
        
        for aug_text in aug_texts:
            if needed <= 0:
                break
            augmented_rows.append({
                'text': aug_text,
                'label': label_name
            })
            needed -= 1

# Gabungkan ke dataframe asli
df_aug = pd.DataFrame(augmented_rows)
df = pd.concat([df, df_aug], ignore_index=True)

# VERIFIKASI

print("\n=== DISTRIBUSI SETELAH AUGMENTASI ===")
print(df['label'].value_counts())
print(f"\nTotal data: {df.shape[0]}")

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Distyy\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Distyy\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!



Geografi: 19 → target 200 (perlu tambah 181)

Demografi: 60 → target 200 (perlu tambah 140)

=== DISTRIBUSI SETELAH AUGMENTASI ===
label
Politik                    2949
Sosial Budaya               416
Ideologi                    339
Pertahanan dan Keamanan     330
Ekonomi                     308
Demografi                   200
Geografi                    200
Sumber Daya Alam            153
Name: count, dtype: int64

Total data: 4895


PREPROCESSING

In [26]:
# Cek apakah Sastrawi sudah bisa diimport
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

factory = StemmerFactory()
stemmer = factory.create_stemmer()

stop_factory = StopWordRemoverFactory()
stopword = stop_factory.create_stop_word_remover()

# Test
test = "membangun negeri yang lebih baik"
print(stemmer.stem(test))
print(stopword.remove(test))

bangun negeri yang lebih baik
membangun negeri lebih baik


In [27]:
# Cek struktur slang_indo.csv
slang_df = pd.read_csv('slang_indo.csv')
print(slang_df.head(10))
print(f"\nShape: {slang_df.shape}")
print(f"\nKolom: {slang_df.columns.tolist()}")

    aamiin     amin 
0     adek     adik 
1     adlh   adalah 
2      aer      air 
3  aiskrim  es krim 
4       aj     saja 
5      aja     saja 
6     ajah     saja 
7   ajalah     saja 
8      aje     saja 
9      ajh     saja 

Shape: (1317, 2)

Kolom: ['aamiin', 'amin ']


In [28]:
# Load slang dengan perbaikan header
slang_df = pd.read_csv('slang_indo.csv', header=None, names=['slang', 'formal'])

# Bersihkan spasi
slang_df['slang'] = slang_df['slang'].str.strip()
slang_df['formal'] = slang_df['formal'].str.strip()

# Jadikan dictionary
slang_dict = dict(zip(slang_df['slang'], slang_df['formal']))

# Verifikasi
print(f"Total kata slang: {len(slang_dict)}")
print(f"\nSample:")
print(dict(list(slang_dict.items())[:5]))

Total kata slang: 1250

Sample:
{'aamiin': 'amin', 'adek': 'adik', 'adlh': 'adalah', 'aer': 'air', 'aiskrim': 'es krim'}


In [ ]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# Inisialisasi Sastrawi
factory = StemmerFactory()
stemmer = factory.create_stemmer()

stop_factory = StopWordRemoverFactory()
stopword_remover = stop_factory.create_stop_word_remover()

# FUNGSI PREPROCESSING

def normalize_slang(text, slang_dict):
    words = text.split()
    normalized = [slang_dict.get(word, word) for word in words]
    return ' '.join(normalized)

def preprocess_text(text):
    # 1. Normalisasi slang
    text = normalize_slang(text, slang_dict)
    
    # 2. Stopword removal
    text = stopword_remover.remove(text)
    
    # 3. Stemming
    text = stemmer.stem(text)
    
    return text

# APPLY KE DATAFRAME

# Ini proses berat, kasih progress bar
from tqdm import tqdm
tqdm.pandas()

print("Preprocessing dimulai... (ini butuh beberapa menit)")
df['text_processed'] = df['text'].progress_apply(preprocess_text)


print("\n=== SAMPLE HASIL PREPROCESSING ===")
df[['text', 'text_processed', 'label']].sample(5)

Preprocessing dimulai... (ini butuh beberapa menit)


  0%|          | 0/4895 [00:00<?, ?it/s]

100%|██████████| 4895/4895 [24:23<00:00,  3.34it/s] 


=== SAMPLE HASIL PREPROCESSING ===


,text,text_processed,label
392,konsekuensi yang harus ditanggung oleh anies m...,konsekuensi harus tanggung anies meski bukan i...,Politik
1483,pemberdayaan perempuan melalui serat kartini a...,daya perempuan lalu serat kartini bukti peduli...,Sosial Budaya
4003,calon presiden capres nomor ururt 2 prabowo su...,calon presiden capres nomor ururt 2 prabowo su...,Politik
3445,gamungkin banget 02 punya rasa peduli sama lin...,gamungkin sekali 02 punya rasa peduli sama lin...,Sumber Daya Alam
876,capres 01 anies baswedan mengaku sudah mengant...,capres 01 anies baswedan aku kantong potensi k...,Sumber Daya Alam


In [30]:
# 1. Cek ada tidak hasil preprocessing yang kosong
print(f"Teks kosong setelah preprocessing: {(df['text_processed'].str.strip() == '').sum()}")

# 2. Cek panjang teks setelah preprocessing
print(f"\nDistribusi panjang teks:")
print(df['text_processed'].str.len().describe())

# 3. Cek distribusi label masih aman
print(f"\nDistribusi label:")
print(df['label'].value_counts())

Teks kosong setelah preprocessing: 0

Distribusi panjang teks:
count    4895.000000
mean      145.332380
std        67.210221
min         7.000000
25%        99.000000
50%       141.000000
75%       176.000000
max       483.000000
Name: text_processed, dtype: float64

Distribusi label:
label
Politik                    2949
Sosial Budaya               416
Ideologi                    339
Pertahanan dan Keamanan     330
Ekonomi                     308
Demografi                   200
Geografi                    200
Sumber Daya Alam            153
Name: count, dtype: int64


In [31]:
# Simpan hasil preprocessing supaya tidak perlu ulang dari awal
df.to_csv('data_preprocessed.csv', index=False)
print("Data berhasil disimpan!")

Data berhasil disimpan!


TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# LABEL ENCODING

le = LabelEncoder()
df['label_encoded'] = le.fit_transform(df['label'])

print("Mapping label:")
for i, label in enumerate(le.classes_):
    print(f"  {i} → {label}")

# SPLIT DATA

X = df['text_processed']
y = df['label_encoded']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"\nUkuran data:")
print(f"X_train : {X_train.shape[0]} baris")
print(f"X_test  : {X_test.shape[0]} baris")

# TF-IDF

tfidf = TfidfVectorizer(
    max_features = 10000,
    ngram_range  = (1, 2),
    min_df       = 2,
    sublinear_tf = True
)

# Fit HANYA di training data, transform keduanya
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print(f"\nHasil TF-IDF:")
print(f"Shape X_train : {X_train_tfidf.shape}")
print(f"Shape X_test  : {X_test_tfidf.shape}")

Mapping label:
  0 → Demografi
  1 → Ekonomi
  2 → Geografi
  3 → Ideologi
  4 → Pertahanan dan Keamanan
  5 → Politik
  6 → Sosial Budaya
  7 → Sumber Daya Alam

Ukuran data:
X_train : 3916 baris
X_test  : 979 baris

Hasil TF-IDF:
Shape X_train : (3916, 10000)
Shape X_test  : (979, 10000)


In [33]:
import pickle

# Simpan tfidf vectorizer supaya bisa dipakai untuk prediksi data unlabel nanti
with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

# Simpan label encoder juga
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

print("TF-IDF dan Label Encoder berhasil disimpan!")

TF-IDF dan Label Encoder berhasil disimpan!


In [34]:
import xgboost
print(xgboost.__version__)

3.2.0


TRAINING


In [ ]:
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
from xgboost import XGBClassifier
import time

# DEFINISI MODEL

models = {
    'SVM': LinearSVC(
        class_weight='balanced',
        max_iter=1000,
        random_state=42
    ),
    'Logistic Regression': LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=42
    ),
    'XGBoost': XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        use_label_encoder=False,
        eval_metric='mlogloss',
        random_state=42
    )
}

# TRAINING + EVALUASI

results = {}

for name, model in models.items():
    print(f"\n{'='*40}")
    print(f"Training {name}...")
    
    start = time.time()
    model.fit(X_train_tfidf, y_train)
    elapsed = time.time() - start
    
    # Prediksi
    y_pred = model.predict(X_test_tfidf)
    
    # Metric
    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred, average='macro')
    
    results[name] = {
        'accuracy': acc,
        'f1_macro': f1,
        'time'    : elapsed
    }
    
    print(f"Waktu training : {elapsed:.1f} detik")
    print(f"Akurasi        : {acc:.4f} ({acc*100:.2f}%)")
    print(f"F1-macro       : {f1:.4f} ({f1*100:.2f}%)")

# RINGKASAN

print(f"\n{'='*40}")
print("RINGKASAN PERBANDINGAN MODEL")
print(f"{'='*40}")
print(f"{'Model':<25} {'Akurasi':>10} {'F1-Macro':>10} {'Waktu':>8}")
print(f"{'-'*55}")
for name, res in results.items():
    print(f"{name:<25} {res['accuracy']*100:>9.2f}% {res['f1_macro']*100:>9.2f}% {res['time']:>6.1f}s")


Training SVM...
Waktu training : 0.2 detik
Akurasi        : 0.7640 (76.40%)
F1-macro       : 0.6844 (68.44%)

Training Logistic Regression...
Waktu training : 0.7 detik
Akurasi        : 0.6987 (69.87%)
F1-macro       : 0.6633 (66.33%)

Training XGBoost...


d:\ANACONDA\Lib\site-packages\xgboost\training.py:200: UserWarning: [17:41:04] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Waktu training : 34.5 detik
Akurasi        : 0.7579 (75.79%)
F1-macro       : 0.6272 (62.72%)

RINGKASAN PERBANDINGAN MODEL
Model                        Akurasi   F1-Macro    Waktu
-------------------------------------------------------
SVM                           76.40%     68.44%    0.2s
Logistic Regression           69.87%     66.33%    0.7s
XGBoost                       75.79%     62.72%   34.5s


SVM + TUNING TF-IDF

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold

# PIPELINE TF-IDF + SVM

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(sublinear_tf=True)),
    ('svm', LinearSVC(class_weight='balanced', random_state=42))
])

# PARAMETER GRID

param_grid = {
    'tfidf__max_features': [10000, 20000],
    'tfidf__ngram_range' : [(1,1), (1,2)],
    'tfidf__min_df'      : [1, 2],
    'svm__C'             : [0.1, 1, 10]
}

# GRIDSEARCH + KFOLD

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1,          # pakai semua core CPU
    verbose=2
)

print("GridSearch dimulai... (ini butuh beberapa menit)")
grid_search.fit(X_train, y_train)

# HASIL

print(f"\n{'='*40}")
print(f"Best Parameter : {grid_search.best_params_}")
print(f"Best CV Score  : {grid_search.best_score_*100:.2f}%")

# Evaluasi di test set
y_pred_best = grid_search.predict(X_test)
acc_best = accuracy_score(y_test, y_pred_best)
f1_best  = f1_score(y_test, y_pred_best, average='macro')

print(f"\nHasil di Test Set:")
print(f"Akurasi  : {acc_best*100:.2f}%")
print(f"F1-macro : {f1_best*100:.2f}%")

print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_best, target_names=le.classes_))

GridSearch dimulai... (ini butuh beberapa menit)
Fitting 5 folds for each of 24 candidates, totalling 120 fits

Best Parameter : {'svm__C': 1, 'tfidf__max_features': 20000, 'tfidf__min_df': 1, 'tfidf__ngram_range': (1, 2)}
Best CV Score  : 76.99%

Hasil di Test Set:
Akurasi  : 76.92%
F1-macro : 68.72%

Classification Report:
                         precision    recall  f1-score   support

              Demografi       0.90      0.88      0.89        40
                Ekonomi       0.72      0.68      0.70        62
               Geografi       0.97      0.97      0.97        40
               Ideologi       0.69      0.54      0.61        68
Pertahanan dan Keamanan       0.72      0.70      0.71        66
                Politik       0.82      0.87      0.84       590
          Sosial Budaya       0.41      0.40      0.40        83
       Sumber Daya Alam       0.50      0.30      0.38        30

               accuracy                           0.77       979
              macro a

In [37]:
import pickle

best_model = grid_search.best_estimator_

with open('svm_baseline.pkl', 'wb') as f:
    pickle.dump(best_model, f)

print("Model baseline tersimpan!")

Model baseline tersimpan!


UNDERSAMPLING

In [38]:
# Cek dari mana undersampling dilakukan
# Harus dari X_train, BUKAN dari df keseluruhan!
print(f"Politik di X_train: {(y_train == 5).sum()}")
# label 5 = Politik (dari mapping LabelEncoder kita)

Politik di X_train: 2359


In [ ]:
import numpy as np

# UNDERSAMPLING POLITIK

# Pisahkan index Politik dan non-Politik di training
idx_politik     = np.where(y_train == 5)[0]
idx_non_politik = np.where(y_train != 5)[0]

print(f"Index Politik    : {len(idx_politik)}")
print(f"Index Non-Politik: {len(idx_non_politik)}")

# Random sample 800 dari Politik
np.random.seed(42)
idx_politik_sample = np.random.choice(idx_politik, size=800, replace=False)

# Gabungkan
idx_final = np.concatenate([idx_politik_sample, idx_non_politik])

# Apply ke data training
X_train_arr = np.array(X_train)  # convert ke array dulu
y_train_arr = np.array(y_train)

X_train_under = X_train_arr[idx_final]
y_train_under = y_train_arr[idx_final]

print(f"\nDistribusi setelah undersampling:")
unique, counts = np.unique(y_train_under, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  {le.classes_[u]:<25}: {c}")

print(f"\nTotal X_train baru: {len(X_train_under)}")

Index Politik    : 2359
Index Non-Politik: 1557

Distribusi setelah undersampling:
  Demografi                : 160
  Ekonomi                  : 246
  Geografi                 : 160
  Ideologi                 : 271
  Pertahanan dan Keamanan  : 264
  Politik                  : 800
  Sosial Budaya            : 333
  Sumber Daya Alam         : 123

Total X_train baru: 2357


In [ ]:

# TF-IDF ULANG DARI TEKS

# Pakai parameter terbaik dari GridSearch sebelumnya
tfidf_under = TfidfVectorizer(
    max_features = 20000,
    ngram_range  = (1, 2),
    min_df       = 1,
    sublinear_tf = True
)

# Fit HANYA di training yang sudah undersampled
X_train_under_tfidf = tfidf_under.fit_transform(X_train_under)

# Test set tetap sama, transform saja
X_test_tfidf_under = tfidf_under.transform(X_test)

print(f"Shape X_train : {X_train_under_tfidf.shape}")
print(f"Shape X_test  : {X_test_tfidf_under.shape}")

# TRAINING SVM

from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, classification_report

svm_under = LinearSVC(
    class_weight = 'balanced',
    C            = 1,
    max_iter     = 1000,
    random_state = 42
)

svm_under.fit(X_train_under_tfidf, y_train_under)

# EVALUASI

y_pred_under = svm_under.predict(X_test_tfidf_under)

acc = accuracy_score(y_test, y_pred_under)
f1  = f1_score(y_test, y_pred_under, average='macro')

print(f"\n{'='*40}")
print(f"Akurasi  : {acc*100:.2f}%")
print(f"F1-macro : {f1*100:.2f}%")

print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_under, target_names=le.classes_))

Shape X_train : (2357, 20000)
Shape X_test  : (979, 20000)

Akurasi  : 71.09%
F1-macro : 66.32%

Classification Report:
                         precision    recall  f1-score   support

              Demografi       0.88      0.88      0.88        40
                Ekonomi       0.64      0.71      0.67        62
               Geografi       0.95      0.97      0.96        40
               Ideologi       0.61      0.63      0.62        68
Pertahanan dan Keamanan       0.53      0.74      0.62        66
                Politik       0.86      0.73      0.79       590
          Sosial Budaya       0.34      0.51      0.40        83
       Sumber Daya Alam       0.33      0.40      0.36        30

               accuracy                           0.71       979
              macro avg       0.64      0.70      0.66       979
           weighted avg       0.75      0.71      0.72       979

